In [292]:
import pandas as pd
import csv

In [293]:
df = pd.read_csv('./data/weather-stations/kelowna/kelowna_stations.csv')

In [294]:
df.sort_values(by=['Latitude'], ascending = True)

,Network ID,Network Name,Native ID,Station ID,Station Name,History ID,Province,Longitude,Latitude,Elevation (m),Record Start,Record End,Obs Freq,Variables
78,1,EC,1131410,1112,CARMI,1515,BC,-119.083333,49.500000,1245.0,1924-01-01,1969-03-31,Daily,Precipitation Amount|Rainfall Amount|Snowfall ...
7,1,EC,1126160,243,PENTICTON SEWAGE PLANT,646,BC,-119.600000,49.500000,344.0,1954-10-01,1969-11-30,Daily,Precipitation Amount|Rainfall Amount|Snowfall ...
285,17,ARDA,112254,4998,ROGER RCH DS,6678,BC,-119.777500,49.513333,832.0,1980-10-08,1981-07-22,Daily,Precipitation Amount|Surface Snow Depth (Point)
145,12,FLNRO-WMB,328,1876,PENTICTON RS,2279,BC,-119.553300,49.518300,427.0,1988-08-16,2023-10-08,Hourly,Dew Point Temperature|Precipitation Amount|Pre...
165,12,FLNRO-WMB,393,1936,NICOLL,2339,BC,-118.360300,49.526700,866.0,1988-10-28,2024-01-14,Hourly,Dew Point Temperature|Precipitation Amount|Pre...
486,12,FLNRO-WMB,1180,11000,ZZ RED CREEK TEMP,13168,BC,-120.220833,49.529167,1400.0,2014-06-25,2018-09-11,Unspecified,Precipitation Amount|Relative Humidity (Mean)|...
247,17,ARDA,112154,4956,PENTICTON CK,6636,BC,-119.503333,49.530000,1140.0,1971-04-01,1972-11-02,Daily,Precipitation (Cumulative)|Surface Snow Depth ...
303,17,ARDA,113072,5071,LAKEVALE,6751,BC,-119.076667,49.535000,887.0,1971-04-26,1972-09-29,Irregular,Precipitation (Cumulative)|Surface Snow Depth ...
189,14,ENV-ASP,2E07P,2553,Grano Creek,2956,BC,-118.683000,49.550000,1860.0,1997-07-24,2020-10-01,Daily,Precipitation 1hr|Precipitation Amount|Precipi...
44,1,EC,1125381,1057,NARAMATA,1460,BC,-119.566667,49.550000,411.0,1971-04-01,2004-02-29,Daily,Precipitation Amount|Precipitation Climatology...


In [295]:
total_burn_data = pd.read_csv("./data/satellite-burn/satellite_burn_cleaned.csv")
total_burn_data[:70]

,IGNITION_DATE,FIRE_SIZE_HA,LATITUDE,LONGITUDE
0,2017/01/01,0.009,49.1217,-115.9226
1,2017/01/01,0.009,51.6025,-122.2677
2,2017/02/10,0.009,50.8894,-119.3201
3,2017/03/01,0.009,52.1870,-121.9566
4,2017/03/13,0.750,49.1705,-116.5328
5,2017/04/01,8.700,50.2925,-121.6444
6,2017/04/01,0.000,49.5282,-123.8333
7,2017/04/03,1.200,51.9963,-123.1572
8,2017/04/03,0.009,51.9366,-122.9875
9,2017/04/03,0.009,50.1729,-119.3620


In [296]:
total_burn_data['LATITUDE'] = pd.to_numeric(total_burn_data['LATITUDE'], errors='coerce')
total_burn_data['LONGITUDE'] = pd.to_numeric(total_burn_data['LONGITUDE'], errors='coerce')

total_burn_data.dropna(subset=['LATITUDE', 'LONGITUDE'], inplace=True)

In [313]:
def spatially_split_stations(df, num_long_splits, num_lat_splits):
    longitude_max = df['Longitude'].max()
    longitude_min = df['Longitude'].min()
    latitude_max = df['Latitude'].max()
    latitude_min = df['Latitude'].min()
    longitude_distance = longitude_max - longitude_min
    latitude_distance = latitude_max - latitude_min
    
    long_split_distance = longitude_distance / num_long_splits
    lat_split_distance = latitude_distance / num_lat_splits
    
    long_track = longitude_min
    lat_track = latitude_min
    print(f"min_long = {longitude_min}, min_lat = {latitude_min}, max_long = {longitude_max}, max_lat = {latitude_max}, long_split = {long_split_distance}, lat_split = {lat_split_distance}")
    zero_count = 0
    num_above = 0
    
    for i in range(num_lat_splits):
        lat_track = latitude_min + i * lat_split_distance
        for j in range(num_long_splits):
            long_track = longitude_min + j * long_split_distance
            
            latitude_mask_station = (df['Latitude'] < lat_track + lat_split_distance) & (df['Latitude'] >= lat_track)
            longitude_mask_station = (df['Longitude'] < long_track + long_split_distance) & (df['Longitude'] >= long_track)
            
            latitude_mask_burn = (burn_data['LATITUDE'] < lat_track + lat_split_distance) & (burn_data['LATITUDE'] >= lat_track)
            longitude_mask_burn = (burn_data['LONGITUDE'] < long_track + long_split_distance) & (burn_data['LONGITUDE'] >= long_track)
            
            station_df = df[latitude_mask_station & longitude_mask_station]
            single_burn = burn_data[latitude_mask_burn & longitude_mask_burn]['FIRE_SIZE_HA'].sum(axis=0)
            burn_array_sample.append(single_burn)
            
            print(f"total burn is {single_burn}")
            
            if single_burn >= 0.01: # in order to see number of regions with noticeable burn area
                num_above += 1
                
            if len(station_df) == 0: 
                zero_count += 1
            station_df.to_csv(f"data/weather-stations/kelowna/fine_area_{i*num_long_splits+j+1}_stations.csv")
            
            print(f"num of stations to aggregate is {len(station_df)}")
            
    print(f"zero count is {zero_count}")
    print(f"num burn above 0.01 is {num_above}")

In [314]:
spatially_split_stations(df, 6, 6)

min_long = -120.330555555556, min_lat = 49.5, max_long = -118.193333333333, max_lat = 51.0, long_split = 0.3562037037038337, lat_split = 0.25
total burn is 0.4
num of stations to aggregate is 15
total burn is 2376.863
num of stations to aggregate is 87
total burn is 16.454
num of stations to aggregate is 14
total burn is 1.263
num of stations to aggregate is 11
total burn is 0.21800000000000003
num of stations to aggregate is 6
total burn is 1096.0
num of stations to aggregate is 1
total burn is 5.215
num of stations to aggregate is 6
total burn is 4108.0289999999995
num of stations to aggregate is 18
total burn is 18.107999999999997
num of stations to aggregate is 55
total burn is 1.854
num of stations to aggregate is 10
total burn is 0.036
num of stations to aggregate is 8
total burn is 0.0
num of stations to aggregate is 5
total burn is 9.814
num of stations to aggregate is 8
total burn is 1.9289999999999998
num of stations to aggregate is 5
total burn is 0.166
num of stations to ag

In [315]:
def get_temporal_category(date_string, num_days):
    num = pd.to_datetime(date_string).dayofyear
    temporal_group = (num - 1) // num_days + 1
    return temporal_group

In [316]:
pd.set_option('display.min_rows',300)
pd.set_option('display.max_rows',300)
longitude_max = df['Longitude'].max()
longitude_min = df['Longitude'].min()
latitude_max = df['Latitude'].max()
latitude_min = df['Latitude'].min()

my_filter = (total_burn_data['LATITUDE'] <= latitude_max) & (total_burn_data['LATITUDE'] >= latitude_min)
my_filter2 = (total_burn_data['LONGITUDE'] <= longitude_max) & (total_burn_data['LONGITUDE'] >= longitude_min)
kelowna_burn_data = total_burn_data[my_filter & my_filter2]
kelowna_burn_data[:300]

,IGNITION_DATE,FIRE_SIZE_HA,LATITUDE,LONGITUDE
2,2017/02/10,0.009,50.889400,-119.320100
9,2017/04/03,0.009,50.172900,-119.362000
46,2017/04/21,0.009,49.816600,-119.758600
77,2017/05/02,0.000,49.949400,-119.558300
86,2017/05/05,0.009,49.506200,-119.514800
136,2017/05/22,0.009,50.956600,-120.122500
144,2017/05/23,1.000,50.218300,-119.194800
145,2017/05/23,4.700,50.292600,-118.983000
151,2017/05/23,0.009,50.011500,-119.398600
166,2017/05/25,0.009,50.230500,-119.514400


In [318]:
import os

def spatially_split_burn(all_burn, num_long_splits, num_lat_splits, year_min, year_max, temporal_min, temporal_max, window):
    base_dir = './data/final-burn'
    
    longitude_max = all_burn['LONGITUDE'].max()
    longitude_min = all_burn['LONGITUDE'].min()
    latitude_max = all_burn['LATITUDE'].max()
    latitude_min = all_burn['LATITUDE'].min()
    longitude_distance = longitude_max - longitude_min
    latitude_distance = latitude_max - latitude_min

    long_split_distance = longitude_distance / num_long_splits
    lat_split_distance = latitude_distance / num_lat_splits

    long_track = longitude_min
    lat_track = latitude_min

    for year in range(year_min,year_max):        
        for temporal_split in range(temporal_min,temporal_max):
            segmented_burn = []
            
            burn_filter_year = all_burn['IGNITION_DATE'].str[:4] == str(year)
            burn_filter_temporal = all_burn['IGNITION_DATE'].apply(get_temporal_category, args=(window,)) == temporal_split
            
            for i in range(num_lat_splits):
                lat_track = latitude_min + i * lat_split_distance
                
                for j in range(num_long_splits):
                    long_track = longitude_min + j * long_split_distance

                    latitude_mask_burn = (all_burn['LATITUDE'] < lat_track + lat_split_distance) & (all_burn['LATITUDE'] >= lat_track)
                    longitude_mask_burn = (all_burn['LONGITUDE'] < long_track + long_split_distance) & (all_burn['LONGITUDE'] >= long_track)

                    burn_filter_final = burn_filter_year & burn_filter_temporal & latitude_mask_burn & longitude_mask_burn
                    
                    numerical_size = all_burn[burn_filter_final]['FIRE_SIZE_HA'].sum(axis=0)
                    
                    segmented_burn.append(numerical_size)
            
            curr_dir = base_dir + f'/{year}'
            if not os.path.exists(curr_dir):
                os.makedirs(curr_dir)
            
            df = pd.DataFrame(segmented_burn)
            df.to_csv(curr_dir + f'/group_{temporal_split}.csv', index = False, header = False)

In [319]:
spatially_split_burn(kelowna_burn_data, 6, 6, 2017, 2024, 1, 53, 7)

In [324]:
check_set = set()

def temporal_process(year_min, year_max, termporal_min, temporal_max, fine_split_min, fine_split_max, window):
    weather_dir = './data/final-weather/'
    base_dir = "./data/weather-stations/kelowna/"
    
    for curr_year in range(year_min,year_max):

        for i in range(fine_split_min,fine_split_max):
            filtered_df = pd.read_csv(base_dir + f"fine_area_{i}_stations.csv")        

            for temporal_split in range(termporal_min,temporal_max):
                temporal_dataframes = []

                for index, row in filtered_df.iterrows():
                    network_name = row['Network Name']
                    native_id = row['Native ID']

                    file_path = f'./data/weather/{curr_year}/{network_name}/{native_id}.csv'
                    temp_df, temporal_data = [], []

                    if os.path.exists(file_path):
                        temp_df = pd.read_csv(file_path, skiprows = 1)
                    else:
                        check_set.add((network_name, native_id))
                        continue

                    if len(temp_df) != 0:
                        temporal_data = temp_df[temp_df[' time'].apply(get_temporal_category, args=(window,)) == temporal_split]
                    
                    if len(temporal_data) != 0:
                        temporal_dataframes.append(temporal_data)
                
#                 print(temporal_dataframes)
                
                if temporal_dataframes:
                    combined_df = pd.concat(temporal_dataframes, ignore_index=True, axis=0, join='outer')
                    combined_df.fillna(-999, inplace = True)
                    if not os.path.exists(weather_dir + f'{curr_year}/group_{temporal_split}'):
                        os.makedirs(weather_dir + f'{curr_year}/group_{temporal_split}')
                    combined_df.to_csv(weather_dir + f'{curr_year}/group_{temporal_split}/fine_area_{i}_weather.csv', index=False, header=True)

In [ ]:
temporal_process(2017, 2022, 1, 53, 1, 37, 7)

In [309]:
print(len(check_set))

395


Training Pipeline Begins

In [251]:
import numpy as np

def align_data():
    curr_year = 2017
    weather_process_base_dir = './data/final-weather/2017'
    new_dir = './data/train-weather/2017'
    
    for group in range(1,13):
        single_dfs = []
        fine_area_weather_missing = set()
        for i in range(1,101):
            file_path = weather_process_base_dir + f'/group_{group}/fine_area_{i}_weather.csv'
            df = []
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
            else:
                fine_area_weather_missing.add(i)
                continue
            df.replace('None', np.nan, inplace=True)
            
            df = df.apply(pd.to_numeric, errors='coerce')

#             threshold = len(df) * 0.5
            df.replace(-999, np.nan, inplace=True) 

#             cols_to_drop = df.columns[df.isnan().sum() > threshold]
#             df.drop(cols_to_drop, axis=1, inplace=True)

            # impute remaining missing values with column mean
#             df.fillna(df.mean(), inplace=True)

            aggregations = {}
            
            for col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    aggregations[col] = 'mean'  
                else:
                    df.drop(col)
            
            df = df.agg(aggregations)
            df = df.to_frame().transpose()
            
            single_dfs.append(df)
        
        if os.path.exists(f'./data/final-burn/{curr_year}/group_{group+1}.csv') and single_dfs:
            combined_df = pd.concat(single_dfs, ignore_index= True)
            threshold = len(combined_df) * 0.8
            cols_to_drop = combined_df.columns[combined_df.isna().sum() > threshold]
#             print(cols_to_drop)
            combined_df.drop(cols_to_drop, axis=1, inplace=True)
            combined_df.fillna(combined_df.mean(), inplace=True)
            combined_df.to_csv(new_dir + f'/group_{group}.csv', index = False)
            
            burn_area_df = pd.read_csv(f'./data/final-burn/{curr_year}/group_{group+1}.csv', header = None, names=['burn_area'])
            for index, row in burn_area_df.iterrows():
                if index+1 in fine_area_weather_missing:
                    burn_area_df.drop(index, inplace = True)
            burn_area_df.to_csv(f'./data/train-burn/{curr_year}/group_{group}.csv', index = False)

In [252]:
pd.set_option('display.min_rows',70)
pd.set_option('display.max_rows',70)
align_data()

/var/folders/jk/lb9w5x9s3dvcp6jxj99nx1vh0000gn/T/ipykernel_90489/859124636.py:15: DtypeWarning: Columns (1,2,4,6,8,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/folders/jk/lb9w5x9s3dvcp6jxj99nx1vh0000gn/T/ipykernel_90489/859124636.py:15: DtypeWarning: Columns (1,2,4,6,8,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/folders/jk/lb9w5x9s3dvcp6jxj99nx1vh0000gn/T/ipykernel_90489/859124636.py:15: DtypeWarning: Columns (1,2,4,6,8,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/folders/jk/lb9w5x9s3dvcp6jxj99nx1vh0000gn/T/ipykernel_90489/859124636.py:15: DtypeWarning: Columns (1,2,4,6,8,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/folders/jk/lb9w5x9s3dvcp6jxj99nx1vh0000gn/T/ipykernel_90489/859124636.py:15: DtypeW

In [107]:
!pip install xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 8.8 MB/s eta 0:00:00:00:0100:01


In [109]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error

for group in range(15,40):
    

train_df = pd.read_csv('./data/train-weather/2017/group_25.csv')
target_df = pd.read_csv('./data/train-burn/2017/group_25.csv')

X_train = train_df.iloc[:45, :]
y_train = target_df.iloc[:45, 0]
X_test = train_df.iloc[45:60, :]
y_test = target_df.iloc[45:60, 0]

model = xgb.XGBRegressor(objective='reg:squarederror')

model.fit(X_train, y_train)

predictions = model.predict(X_test)

for i in range(len(predictions)):
    print(f"Predicted: {predictions[i]}, Actual: {y_test.iloc[i]}")

mse = mean_squared_error(y_test, predictions)
print("Mean Squared Error on Test Data:", mse)

Predicted: 0.8843307495117188, Actual: 0.0
Predicted: 1.675981879234314, Actual: 0.0
Predicted: -0.00029467607964761555, Actual: 0.0
Predicted: 1.675981879234314, Actual: 0.0
Predicted: 1.675981879234314, Actual: 0.0
Predicted: -0.04096442833542824, Actual: 0.0
Predicted: 3.3134653568267822, Actual: 0.0
Predicted: 0.6431534886360168, Actual: 0.0
Predicted: 0.0005261584301479161, Actual: 0.0
Predicted: 0.6079038977622986, Actual: 0.0
Predicted: 0.3213668167591095, Actual: 0.0
Predicted: 1.675981879234314, Actual: 0.0
Predicted: 0.4828042685985565, Actual: 0.0
Predicted: 0.6353285312652588, Actual: 0.0
Predicted: 1.675981879234314, Actual: 0.0
Mean Squared Error on Test Data: 1.8220373856858303


In [ ]:
# import os
# my_filter = ['ONE_DAY_PRECIPITATION', 'ONE_DAY_RAIN', 'MIN_TEMP', 'ONE_DAY_SNOW', 'time', 'MAX_TEMP']
filtered_df = df
for curr_year in range(2012, 2013):
    yearly_dataframes = []

    for index, row in filtered_df.iterrows():
        network_name = row['Network Name']
        native_id = row['Native ID']
        
        file_path = f'./data/weather/{curr_year}/{network_name}/{native_id}.csv'

        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path, skiprows=1)
            
            yearly_dataframes.append(temp_df)

    if yearly_dataframes:
        combined_df = pd.concat(yearly_dataframes, ignore_index=True)
        combined_df.columns = [i.lstrip() for i in combined_df.columns]
#         existing_columns = [c for c in my_filter if c in combined_df.columns]
#         combined_df = combined_df[existing_columns]
#       print(combined_df.columns)
        combined_csv_path = f'./training_data_20123/combined_data_{curr_year}.csv'
        combined_df.to_csv(combined_csv_path, index=False)
        print(combined_df)